In [1]:
# ── CELL 1: Import Libraries ──────────────────────
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score,
                             classification_report)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")

All libraries imported successfully


In [2]:
# ── CELL 2: Reload All Preprocessed Datasets ──────

# ── ADULT ──────────────────────────────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age','workclass','fnlwgt','education','education-num',
           'marital-status','occupation','relationship','race',
           'sex','capital-gain','capital-loss','hours-per-week',
           'native-country','income']
df_adult = pd.read_csv(url, names=columns, na_values=' ?',
                        skipinitialspace=True).dropna()
df_adult['income_binary'] = (df_adult['income']=='>50K').astype(int)
df_adult['sex_binary'] = (df_adult['sex']=='Male').astype(int)
cat_cols = ['workclass','education','marital-status','occupation',
            'relationship','race','native-country']
le = LabelEncoder()
for col in cat_cols:
    df_adult[col] = le.fit_transform(df_adult[col].astype(str))
df_adult = df_adult.drop(['income','sex','fnlwgt'], axis=1)
X_adult = df_adult.drop('income_binary', axis=1)
y_adult = df_adult['income_binary']
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_adult, y_adult, test_size=0.2, random_state=42, stratify=y_adult)
scaler = StandardScaler()
X_train_a = pd.DataFrame(scaler.fit_transform(X_train_a), columns=X_adult.columns)
X_test_a = pd.DataFrame(scaler.transform(X_test_a), columns=X_adult.columns)

# ── GERMAN CREDIT ──────────────────────────────────
url_g = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"
col_names = ['status','duration','credit_history','purpose','credit_amount',
             'savings','employment','installment_rate','personal_status_sex',
             'other_debtors','residence','property','age','other_installments',
             'housing','existing_credits','job','dependents','telephone',
             'foreign_worker','target']
df_german = pd.read_csv(url_g, sep=' ', header=None, names=col_names)
gender_map = {'A91':'Male','A92':'Female','A93':'Male','A94':'Male','A95':'Female'}
df_german['gender_binary'] = (df_german['personal_status_sex'].map(gender_map)=='Male').astype(int)
df_german['target_binary'] = (df_german['target']==2).astype(int)
cat_cols_g = ['status','credit_history','purpose','savings','employment',
              'personal_status_sex','other_debtors','property',
              'other_installments','housing','job','telephone','foreign_worker']
for col in cat_cols_g:
    df_german[col] = le.fit_transform(df_german[col].astype(str))
df_german = df_german.drop('target', axis=1)
X_german = df_german.drop('target_binary', axis=1)
y_german = df_german['target_binary']
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_german, y_german, test_size=0.2, random_state=42, stratify=y_german)
scaler_g = StandardScaler()
X_train_g = pd.DataFrame(scaler_g.fit_transform(X_train_g), columns=X_german.columns)
X_test_g = pd.DataFrame(scaler_g.transform(X_test_g), columns=X_german.columns)

# ── LAW SCHOOL ─────────────────────────────────────
df_law = pd.read_csv(r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\data\law_school\law_dataset.csv')
df_law = df_law.dropna(subset=['pass_bar','male','racetxt','lsat','ugpa'])
le_race = LabelEncoder()
df_law['race_encoded'] = le_race.fit_transform(df_law['racetxt'].astype(str))
features = ['lsat','ugpa','zfygpa','zgpa','fulltime','fam_inc',
            'male','race_encoded','tier']
df_law_clean = df_law[features+['pass_bar']].dropna()
X_law = df_law_clean[features]
y_law = df_law_clean['pass_bar'].astype(int)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_law, y_law, test_size=0.2, random_state=42, stratify=y_law)
scaler_l = StandardScaler()
X_train_l = pd.DataFrame(scaler_l.fit_transform(X_train_l), columns=X_law.columns)
X_test_l = pd.DataFrame(scaler_l.transform(X_test_l), columns=X_law.columns)

print("All datasets reloaded and ready")
print(f"Adult:         Train {X_train_a.shape} | Test {X_test_a.shape}")
print(f"German Credit: Train {X_train_g.shape} | Test {X_test_g.shape}")
print(f"Law School:    Train {X_train_l.shape} | Test {X_test_l.shape}")

All datasets reloaded and ready
Adult:         Train (26048, 13) | Test (6513, 13)
German Credit: Train (800, 21) | Test (200, 21)
Law School:    Train (14953, 9) | Test (3739, 9)


In [3]:
# ── CELL 3: Evaluation Function ───────────────────
def evaluate_model(model, X_train, X_test, y_train, y_test, 
                   model_name, dataset_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1] if hasattr(
        model, 'predict_proba') else None
    
    results = {
        'Model': model_name,
        'Dataset': dataset_name,
        'Accuracy': round(accuracy_score(y_test, y_pred)*100, 2),
        'Precision': round(precision_score(y_test, y_pred)*100, 2),
        'Recall': round(recall_score(y_test, y_pred)*100, 2),
        'F1': round(f1_score(y_test, y_pred)*100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, y_prob)*100, 2) 
                   if y_prob is not None else 'N/A'
    }
    return results

print("Evaluation function ready")

Evaluation function ready


In [4]:
# ── CELL 4: Train All 4 Models on All 3 Datasets ──
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM':                 SVC(random_state=42, probability=True),
    'XGBoost':             XGBClassifier(random_state=42, eval_metric='logloss',
                                         verbosity=0)
}

datasets = {
    'Adult':         (X_train_a, X_test_a, y_train_a, y_test_a),
    'German Credit': (X_train_g, X_test_g, y_train_g, y_test_g),
    'Law School':    (X_train_l, X_test_l, y_train_l, y_test_l)
}

all_results = []

for dataset_name, (X_train, X_test, y_train, y_test) in datasets.items():
    print(f"\nTraining on {dataset_name}...")
    for model_name, model in models.items():
        print(f"  Running {model_name}...", end='')
        result = evaluate_model(model, X_train, X_test,
                                y_train, y_test,
                                model_name, dataset_name)
        all_results.append(result)
        print(f" Accuracy: {result['Accuracy']}%")

print("\n✅ ALL MODELS TRAINED SUCCESSFULLY")


Training on Adult...
  Running Logistic Regression... Accuracy: 82.8%
  Running Random Forest... Accuracy: 85.03%
  Running SVM... Accuracy: 85.58%
  Running XGBoost... Accuracy: 87.27%

Training on German Credit...
  Running Logistic Regression... Accuracy: 77.5%
  Running Random Forest... Accuracy: 78.0%
  Running SVM... Accuracy: 77.0%
  Running XGBoost... Accuracy: 76.5%

Training on Law School...
  Running Logistic Regression... Accuracy: 91.15%
  Running Random Forest... Accuracy: 90.75%
  Running SVM... Accuracy: 91.25%
  Running XGBoost... Accuracy: 90.53%

✅ ALL MODELS TRAINED SUCCESSFULLY


In [5]:
# ── CELL 5: Results Table ─────────────────────────
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(['Dataset','Accuracy'], 
                                     ascending=[True, False])

print("=== BASELINE MODEL RESULTS ===\n")
print(results_df.to_string(index=False))

# Save results
import os
results_df.to_csv(
    r'C:\Users\niyat\Desktop\pr\ML-Bias-Fairness\results\baseline_results.csv',
    index=False)
print("\nResults saved to results folder")

=== BASELINE MODEL RESULTS ===

              Model       Dataset  Accuracy  Precision  Recall    F1  ROC-AUC
            XGBoost         Adult     87.27      76.56   67.92 71.98    92.93
                SVM         Adult     85.58      76.95   57.27 65.67    89.76
      Random Forest         Adult     85.03      71.25   63.39 67.09    89.82
Logistic Regression         Adult     82.80      72.54   45.98 56.28    86.05
      Random Forest German Credit     78.00      71.05   45.00 55.10    80.20
Logistic Regression German Credit     77.50      64.15   56.67 60.18    78.85
                SVM German Credit     77.00      66.67   46.67 54.90    79.30
            XGBoost German Credit     76.50      64.44   48.33 55.24    78.33
                SVM    Law School     91.25      91.54   99.50 95.35    73.96
Logistic Regression    Law School     91.15      92.11   98.64 95.26    87.34
      Random Forest    Law School     90.75      92.31   97.89 95.02    85.56
            XGBoost    Law Schoo